# Download EarthCARE Data

Download EarthCARE Level 2 products from the ESA MAAP catalog using `earthcarekit`.

**Prerequisites:**
- An ESA EO Sign-In account
- A MAAP access token from https://portal.maap.eo.esa.int/earthcare/ ([TOOLS] -> [Generate Data Access Token])
- `earthcarekit` configured with your token (see setup cell below)

## Setup

In [14]:
import earthcarekit as eck

# earthcarekit reads config from ~/.config/earthcarekit/default_config.toml
# If your config file is elsewhere, point to it:
# eck.set_config("/path/to/your/config.toml")

# First time setup:
# 1. eck.create_example_config()
# 2. Get a MAAP token from https://portal.maap.eo.esa.int/earthcare/
#    [TOOLS] -> [Generate Data Access Token] (valid 90 days)
# 3. eck.set_config_maap_token("your_token_here")
#    eck.set_config_to_maap()

## Browse available product types

In [15]:
from pystac_client import Client

CATALOG_URL = "https://catalog.maap.eo.esa.int/catalogue"
COLLECTION_ID = "EarthCAREL2Validated"

catalog = Client.open(CATALOG_URL)
search = catalog.search(collections=[COLLECTION_ID], max_items=200)
items = list(search.items())

product_types = {}
for item in items:
    pt = item.properties.get("product:type", "unknown")
    instruments = item.properties.get("instruments", ["unknown"])
    instrument = instruments[0] if isinstance(instruments, list) else instruments
    geom_type = item.geometry["type"]
    if pt not in product_types:
        product_types[pt] = {"instrument": instrument, "geometry": geom_type, "count": 0}
    product_types[pt]["count"] += 1

print(f"Found {len(product_types)} product types:\n")
for pt, info in sorted(product_types.items()):
    print(f"  {pt:15s}  instrument={info['instrument']:10s}  geometry={info['geometry']:12s}  count={info['count']}")

Found 18 product types:

  AC__TC__2B       instrument=SYNERGETIC  geometry=LineString    count=10
  AM__ACD_2B       instrument=SYNERGETIC  geometry=LineString    count=11
  AM__CTH_2B       instrument=SYNERGETIC  geometry=LineString    count=10
  ATL_AER_2A       instrument=ATLID       geometry=LineString    count=14
  ATL_ALD_2A       instrument=ATLID       geometry=LineString    count=13
  ATL_CTH_2A       instrument=ATLID       geometry=LineString    count=10
  ATL_EBD_2A       instrument=ATLID       geometry=LineString    count=9
  ATL_FM__2A       instrument=ATLID       geometry=LineString    count=9
  ATL_ICE_2A       instrument=ATLID       geometry=LineString    count=10
  ATL_TC__2A       instrument=ATLID       geometry=LineString    count=17
  BM__RAD_2B       instrument=SYNERGETIC  geometry=Polygon       count=15
  CPR_CD__2A       instrument=CPR         geometry=LineString    count=10
  CPR_CLD_2A       instrument=CPR         geometry=LineString    count=12
  CPR_FMR_2A   

## Download orbit 6109D over Norway

Orbit 6109D crosses the Norwegian coast (lon -1 to 19E, lat 22-68N).
We download MSI (2D swath) and ATLID (profile) products.

In [16]:
ORBIT = "06109D"

download_types = [
    "MSI_CM__2A",   # Cloud mask (2D swath)
    "MSI_COP_2A",   # Cloud optical properties (2D swath)
    "MSI_AOT_2A",   # Aerosol optical thickness (2D swath)
    "ATL_AER_2A",   # ATLID aerosol profiles
]

for file_type in download_types:
    print(f"\n--- {file_type} orbit {ORBIT} ---")
    try:
        eck.ecdownload(file_type=file_type, orbit_and_frame=ORBIT)
        print("  OK")
    except Exception as e:
        print(f"  Error: {e}")


--- MSI_CM__2A orbit 06109D ---
#======================================================================#
#                       EarthCARE Download Tool                        #
#                         earthcarekit 0.15.2                          #
#======================================================================#
# Settings
# - is_download=True
# - is_overwrite=False
# - is_unzip=True
# - is_delete=True
# - is_create_subdirs=True
# - is_log=False
# - is_debug=False
# - is_export_results=False
# - idx_selected_input=None
# - config_filepath=</Users/annef/.config/earthcarekit/default_config.toml>
# - data_dirpath=<data>

+----------------------------------------------------------------------+
| STEP 1/2 - Search products                       2026-04-05 17:27:03 |
+----------------------------------------------------------------------+

*[1/1] Search request: MSI_CM__2A, frame=D, orbits=[6109]
 [1/1] Files found in collection 'EarthCAREL2Validated_MAAP': 1

List of files found 

## Verify downloads

In [17]:
for file_type in download_types:
    try:
        result = eck.search_product(file_type=file_type, orbit_and_frame=ORBIT)
        print(f"{file_type}: {result.filepath[0]}")
    except Exception as e:
        print(f"{file_type}: NOT FOUND ({e})")

MSI_CM__2A: /Users/annef/Documents/FAIR2Adapt/DGGS/earthcare-dggs/notebooks/data/level2a/MSI_CM__2A/2025/06/25/BA/ECA_EXBA_MSI_CM__2A_20250625T135845Z_20250929T143920Z_06109D/ECA_EXBA_MSI_CM__2A_20250625T135845Z_20250929T143920Z_06109D.h5
MSI_COP_2A: /Users/annef/Documents/FAIR2Adapt/DGGS/earthcare-dggs/notebooks/data/level2a/MSI_COP_2A/2025/06/25/BA/ECA_EXBA_MSI_COP_2A_20250625T135845Z_20250929T143920Z_06109D/ECA_EXBA_MSI_COP_2A_20250625T135845Z_20250929T143920Z_06109D.h5
MSI_AOT_2A: /Users/annef/Documents/FAIR2Adapt/DGGS/earthcare-dggs/notebooks/data/level2a/MSI_AOT_2A/2025/06/25/BA/ECA_EXBA_MSI_AOT_2A_20250625T135845Z_20250929T212718Z_06109D/ECA_EXBA_MSI_AOT_2A_20250625T135845Z_20250929T212718Z_06109D.h5
ATL_AER_2A: /Users/annef/Documents/FAIR2Adapt/DGGS/earthcare-dggs/notebooks/data/level2a/ATL_AER_2A/2025/06/25/BA/ECA_EXBA_ATL_AER_2A_20250625T135845Z_20250929T172314Z_06109D/ECA_EXBA_ATL_AER_2A_20250625T135845Z_20250929T172314Z_06109D.h5
